In [56]:
from cmdstanpy import CmdStanModel
import arviz as az
from sampler import CycloneDataSimulator
import pandas as pd
import numpy as np

In [58]:
clean_data = pd.read_csv('../processed_data/processed_data.csv')
clean_data.head()
basin_dict = {'AL': 1, 'CP' : 2, 'EP' : 3}
clean_data['basin_int'] = [basin_dict[el] for el in clean_data['Basin']]
basins = clean_data['basin_int'].values
W = np.log(clean_data['VMAX36'].values)
X = clean_data[['VMAX12','MSLP12','POT12','VMPI12','ONI']].values

In [ ]:
stan_file = 'conjugate_model.stan'
model = CmdStanModel(stan_file = stan_file)

# prior parameters (for sampling data)
a = 3
y_alpha = 3
z_alpha = 2
y_gamma = 3
z_gamma = 2 
v2 = 2
y_lam = 3
z_lam = 1

# model parameters
B = 3               # number of ocean basins
D = 5               # number of predictors

# load real data into a Stan dictionary
stan_basins = basins            # Stan is 1-indexed!!!
stan_data = {
    "N" : len(X), "B" : B, "D" : D,
    "W" : W, "X" : X, "basins" : stan_basins,
    "a" : a,
    "y_alpha" : y_alpha, "z_alpha" : z_alpha,
    "y_gamma" : y_gamma, "z_gamma" : z_gamma,
    "v2" : v2,
    "y_lambda" : y_lam, "z_lambda" : z_lam,
}

fit = model.sample(data = stan_data, chains=4, iter_sampling=1000, iter_warmup=1000)
idata = az.from_cmdstanpy(posterior=fit, log_likelihood='log_lik')

16:36:54 - cmdstanpy - INFO - CmdStan start processing


chain 1 |          | 00:00 Status

chain 2 |          | 00:00 Status

chain 3 |          | 00:00 Status

chain 4 |          | 00:00 Status

In [ ]:
az.waic(idata)

In [ ]:
az.loo(idata)

In [ ]:
stan_file = 'conjugate_model_tau2cauchy.stan'
model = CmdStanModel(stan_file = stan_file)

fit = model.sample(data = stan_data, chains=4, iter_sampling=1000, iter_warmup=1000)
idata1 = az.from_cmdstanpy(posterior=fit, log_likelihood='log_lik')

In [ ]:
az.waic(idata1)

In [ ]:
az.loo(idata1)

In [ ]:
stan_file = 'conjugate_model-sig2cauchy.stan'
model = CmdStanModel(stan_file = stan_file)

fit = model.sample(data = stan_data, chains=4, iter_sampling=1000, iter_warmup=1000)
idata2 = az.from_cmdstanpy(posterior=fit, log_likelihood='log_lik')

In [ ]:
az.waic(idata2)

In [ ]:
az.loo(idata2)

In [ ]:
stan_file = 'conjugate_model-allcauchy.stan'
model = CmdStanModel(stan_file = stan_file)

fit = model.sample(data = stan_data, chains=4, iter_sampling=1000, iter_warmup=1000)
idata3 = az.from_cmdstanpy(posterior=fit, log_likelihood='log_lik')

In [ ]:
az.waic(idata3)

In [ ]:
az.loo(idata3)